# Phase 6 : Modélisation Statistique Temporelle (SARIMA & Prophet)

Ce notebook se concentre sur les approches de modélisation statistique classiques :
1. **SARIMA** (modélisation autorégressive avec saisonnalité et différenciation).
2. **Prophet** (modèle additif basé sur la décomposition tendance + saisonnalité de Meta).

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd

sys.path.append(os.path.abspath('..'))
from src.config import DATA_DIR, TARGET_COL
import src.metrics as metrics_mod
from src.models.sarima import SarimaModel
from src.models.prophet import ProphetModel

## 1. Chargement des Séries Univariées

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, 'merged_tourism_data_final.csv'))
df['Date'] = pd.to_datetime(df['Date'])

df_ml = df.dropna(subset=[TARGET_COL]).copy()
train_df = df_ml[df_ml['Date'] <= '2022-12-31']
test_df = df_ml[df_ml['Date'] >= '2023-01-01']

y_train = train_df[TARGET_COL]
y_test = test_df[TARGET_COL]
print(f"Test set length : {len(y_test)} mois")

## 2. Ajustement SARIMA et Prophet

In [ ]:
predictions = {}

print("Modélisation SARIMA...")
sarima = SarimaModel(order=(2, 1, 0), seasonal_order=(1, 0, 1, 12))
sarima.fit(y_train)
predictions['SARIMA'] = sarima.predict(steps=len(y_test))

print("Modélisation Prophet...")
prophet = ProphetModel()
prophet.fit(train_df, target_col=TARGET_COL)
predictions['Prophet'] = prophet.predict(test_df)

results_df = metrics_mod.compare_models(predictions, y_test)
display(results_df)